# Structron — Agentic H-Beam Selection
### An Agentic FEA System for Automated Structural Optimization
**SCIE6063001 — Computational Physics**

This notebook validates the Euler-Bernoulli engine against the manual
reference report (H-Beam 428x407x20x35, 12 m span) and then runs the
agentic selection over 80+ JIS sections to find the lightest section that
passes all six loading cases.

Checks per section: bending stress (sigma = M/Wx <= fy/FoS), deflection
(delta <= L/360) across six load cases, and Euler buckling
(Pcr = pi^2 E I / (KL)^2).

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import structron as st

catalog = st.load_catalog()
print(f'{len(catalog)} H-sections loaded from the JIS catalog')

## 2. Design scenario (reference benchmark)
Q235B steel, 12 m span, operational mass 4320 kg with a x3 load multiplier,
factor of safety 3.0 (allowable stress = fy/FoS).

In [ ]:
scenario = st.Scenario(
    span_mm=12000,
    total_force_n=st.force_from_mass(4320, 3),
    material=st.MATERIALS['Q235B'],
    fos=3.0,
    deflection_limit=360,
)
print(f"Total design force F = {scenario.total_force_n:,.0f} N")
print(f"Allowable stress     = {scenario.material['fy']/scenario.fos:.2f} MPa")
print(f"Allowable deflection = {scenario.span_mm/scenario.deflection_limit:.2f} mm")

## 3. Six-case analysis of the reference section (HW 428x407x20x35)

In [ ]:
ref = next(p for p in catalog if p['name'] == 'HW 428x407x20x35')
rep = st.evaluate_beam(ref, scenario)
df = pd.DataFrame([{
    'Load case': c['label'],
    'Stress (MPa)': round(c['stress_mpa'], 2),
    'Stress %': round(c['stress_ratio'] * 100, 1),
    'Defl (mm)': round(c['deflection_mm'], 2),
    'Defl %': round(c['deflection_ratio'] * 100, 1),
    'Status': 'PASS' if c['passes'] else 'FAIL',
} for c in rep['cases']])
df

### 3a. Error analysis vs the manual report
The engine reproduces the hand calculations to within rounding.

In [ ]:
report_values = {
    'Point load near support': (20.88, 1.80),
    'Point load at 25% span': (51.25, 10.80),
    'Point load at midspan': (68.35, 19.23),
    'Full uniformly distributed load': (34.18, 12.02),
}
rows = []
for c in rep['cases']:
    if c['label'] in report_values:
        rs, rd = report_values[c['label']]
        rows.append({
            'Load case': c['label'],
            'sigma engine': round(c['stress_mpa'], 2), 'sigma report': rs,
            'sigma err %': round(abs(c['stress_mpa'] - rs) / rs * 100, 3),
            'delta engine': round(c['deflection_mm'], 2), 'delta report': rd,
            'delta err %': round(abs(c['deflection_mm'] - rd) / rd * 100, 3),
        })
pd.DataFrame(rows)

## 4. Visualization

In [ ]:
labels = [c['label'] for c in rep['cases']]
x = np.arange(len(labels))
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(x - 0.2, [c['stress_ratio'] * 100 for c in rep['cases']], 0.4, label='Stress %')
ax.bar(x + 0.2, [c['deflection_ratio'] * 100 for c in rep['cases']], 0.4, label='Deflection %')
ax.axhline(100, color='red', ls='--', lw=1, label='Limit (100%)')
ax.set_ylabel('Utilisation (%)')
ax.set_title('HW 428x407x20x35 - utilisation by load case')
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=25, ha='right', fontsize=8)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Deflection profile under full UDL: delta(x) = w x (L^3 - 2 L x^2 + x^3) / (24 E I)
L = scenario.span_mm
E = scenario.material['E']
I = ref['Ix'] * 1e4
w = scenario.total_force_n / L
xs = np.linspace(0, L, 200)
ys = (w * xs / (24 * E * I)) * (L**3 - 2 * L * xs**2 + xs**3)
plt.figure(figsize=(10, 3))
plt.plot(xs / 1000, ys, label='Deflection')
plt.axhline(L / scenario.deflection_limit, color='red', ls='--', label='Allowable L/360')
plt.gca().invert_yaxis()
plt.xlabel('Position along span (m)')
plt.ylabel('Deflection (mm)')
plt.title('Deflection profile under full UDL')
plt.legend()
plt.tight_layout()
plt.show()

## 5. Agentic selection over 80+ candidates
The selector evaluates every catalog section across all six cases and returns
the lightest one that passes - the automated structural optimization step.

In [ ]:
result = st.select_beam(scenario, catalog)
rec = result['recommended']
print(f"Optimal (lightest passing): {rec['profile']['name']} - {rec['profile']['weight']} kg/m")
print(f"Reference section:          {ref['name']} - {ref['weight']} kg/m")
print(f"Mass saving vs reference:   {(1 - rec['profile']['weight'] / ref['weight']) * 100:.1f}%")

rank = pd.DataFrame([{
    'Section': r['profile']['name'],
    'kg/m': r['profile']['weight'],
    'Worst stress %': round(r['worst_stress']['stress_ratio'] * 100, 1),
    'Worst defl %': round(r['worst_deflection']['deflection_ratio'] * 100, 1),
    'Status': 'PASS' if r['passes'] else 'FAIL',
} for r in result['reports'][:15]])
rank

In [ ]:
passing = [r for r in result['reports'] if r['passes']][:12]
names = [r['profile']['name'] for r in passing]
weights = [r['profile']['weight'] for r in passing]
colors = ['green' if r['profile']['name'] == rec['profile']['name'] else 'gray' for r in passing]
plt.figure(figsize=(10, 4))
plt.barh(names, weights, color=colors)
plt.xlabel('Weight (kg/m)')
plt.title('Passing sections by weight (green = optimal)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 6. Finite element validation

The closed-form formulas above are only exact for these idealised cases. The
real engine is a **direct-stiffness FE solver** (`fea.py`). Here we confirm it
reproduces the analytical result, then use it for a portal frame the closed
form cannot handle.

In [ ]:
import fea
E = scenario.material['E']
I = ref['Ix'] * 1e4; A = ref['area'] * 1e2; Sx = ref['Sx'] * 1e3
rows = []
for load, cf in [('udl', 'udl-full'), ('point-mid', 'point-mid')]:
    m = fea.simply_supported_beam(scenario.span_mm, scenario.total_force_n, E, I, A, Sx, n_elems=12, load=load)
    fr = fea.analyze(m)
    c = next(x for x in rep['cases'] if x['id'] == cf)
    rows.append({'Load': load,
                 'FE delta (mm)': round(fr['max_deflection_mm'], 3),
                 'Closed-form (mm)': round(c['deflection_mm'], 3),
                 'FE sigma (MPa)': round(fr['max_stress_mpa'], 3),
                 'Closed-form (MPa)': round(c['stress_mpa'], 3)})
pd.DataFrame(rows)

In [ ]:
# Portal frame (2 columns + beam) - genuine 2D analysis, no closed form
pf = fea.portal_frame(span_mm=10000, height_mm=7000, total_force_n=scenario.total_force_n,
                      E=E, I=I, A=A, Sx=Sx)
pr = fea.analyze(pf)
print(f"Portal frame: {pr['ndof']} DOF, max deflection {pr['max_deflection_mm']:.2f} mm, "
      f"max stress {pr['max_stress_mpa']:.2f} MPa")

## 7. Machine-learning surrogate

We generate a dataset by running the FE solver over thousands of random
designs, then train a neural network to predict the response directly - so
the selector can screen candidates instantly and FE only verifies the pick.

In [ ]:
import surrogate
Xs, ys = surrogate.generate_dataset(1500)
mdl, xsc, ysc, met, (Xte, yte, pred) = surrogate.train_surrogate(Xs, ys)
print(f"Samples: {met['n_train']} train / {met['n_test']} test")
print(f"Deflection: R2 = {met['deflection_r2']:.4f}, MAPE = {met['deflection_mape']:.2f}%")
print(f"Stress:     R2 = {met['stress_r2']:.4f}, MAPE = {met['stress_mape']:.2f}%")

In [ ]:
dt, dp = 10 ** yte[:, 0], 10 ** pred[:, 0]
st, sp = 10 ** yte[:, 1], 10 ** pred[:, 1]
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].scatter(dt, dp, s=8, alpha=0.4)
ax[0].plot([dt.min(), dt.max()], [dt.min(), dt.max()], 'r--')
ax[0].set_xlabel('FE deflection (mm)'); ax[0].set_ylabel('Surrogate'); ax[0].set_title('Deflection parity')
ax[1].scatter(st, sp, s=8, alpha=0.4)
ax[1].plot([st.min(), st.max()], [st.min(), st.max()], 'r--')
ax[1].set_xlabel('FE stress (MPa)'); ax[1].set_ylabel('Surrogate'); ax[1].set_title('Stress parity')
plt.tight_layout(); plt.show()

## 8. Physics-Informed Neural Network (PINN)

As a pure machine-learning-meets-physics showcase, we train a neural network
to satisfy the beam ODE EI w'''' = q directly - no analytical or FE labels.
The loss is the differential-equation residual plus the boundary conditions;
the 4th derivative comes from automatic differentiation.

In [ ]:
import pinn
q = scenario.total_force_n / scenario.span_mm
pmodel, Cconst = pinn.solve_udl(scenario.span_mm, q, E, I, epochs=4000)
pr2 = pinn.evaluate(pmodel, scenario.span_mm, q, E, I)
print(f"PINN  max deflection = {pr2['pinn_max_mm']:.3f} mm")
print(f"Exact max deflection = {pr2['exact_max_mm']:.3f} mm")
print(f"Max error = {pr2['max_abs_err_mm']:.4f} mm ({pr2['max_rel_err_pct']:.2f}%)")

In [ ]:
xm = pr2['xi'] * scenario.span_mm / 1000
plt.figure(figsize=(10, 3))
plt.plot(xm, pr2['w_exact'], lw=2, label='Closed form')
plt.plot(xm, pr2['w_pinn'], '--', label='PINN (trained on the ODE only)')
plt.gca().invert_yaxis()
plt.xlabel('Position (m)'); plt.ylabel('Deflection (mm)')
plt.title('PINN solution vs closed form'); plt.legend()
plt.tight_layout(); plt.show()

## 9. Conclusion

- **Numerical method (FEA):** direct-stiffness solver reproduces the analytical
  benchmark exactly and extends to multi-span beams and portal frames.
- **ML surrogate:** MLP trained on FE data predicts deflection (R2 ~0.99) and
  stress (R2 ~0.997) - instant screening, FE verifies the pick.
- **PINN:** a neural network solves the beam ODE from physics alone, matching
  the closed form to ~0.01%.
- **Agentic + real world:** the tool optimises for cost and availability and an
  agent reads supplier quotations to feed real prices into the search.
- **Scope:** strong-axis bending; LTB, shear and combined axial out of scope.